# Milestone 1: NLP Foundation & Semantic Similarity
This notebook demonstrates baseline text cleaning, TF-IDF vectorization, Word2Vec embeddings, and MAP@3 calculation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load training data
df = pd.read_csv('../data/train.csv')
print('Dataset shape:', df.shape)
df.head(2)

In [ ]:
# Text Preprocessing & Cleaning
def clean_text(text):
    if not isinstance(text, str):
        return ''
    return text.lower().strip()

df['prompt_clean'] = df['prompt'].apply(clean_text)
for c in ['A', 'B', 'C', 'D', 'E']:
    df[c + '_clean'] = df[c].apply(clean_text)

In [ ]:
# TF-IDF Cosine Similarity Baseline
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(df['prompt_clean'])

preds = []
for _, row in df.iterrows():
    p_vec = vectorizer.transform([row['prompt_clean']])
    sims = {}
    for c in ['A', 'B', 'C', 'D', 'E']:
        c_vec = vectorizer.transform([row[c + '_clean']])
        sims[c] = cosine_similarity(p_vec, c_vec)[0][0]
    sorted_choices = sorted(sims, key=sims.get, reverse=True)[:3]
    preds.append(' '.join(sorted_choices))

df['prediction'] = preds
print('Predictions sample:', preds[:5])

In [ ]:
# MAP@3 Metric Function
def map3_eval(df):
    score = 0.0
    for _, row in df.iterrows():
        target = row['answer']
        top_3 = row['prediction'].split()
        if target in top_3:
            rank = top_3.index(target) + 1
            score += 1.0 / rank
    return score / len(df)

print(f'TF-IDF Baseline MAP@3: {map3_eval(df):.4f}')